# 02 · 物理资产构建

模块一真实重建 → VLM 物性估计 → 部件表面重建与双向运动约束 → 多姿态联合分离 → 碰撞代理及动力学验收。

保留模块一原始资产和 start/end 运动范围。重网格仅在各部件内部重建连接；候选几何与关节均须满足输入观测和独立正间隙检查。

In [1]:
import json

from examples._display import setup_notebook, show_animation, show_image, show_metrics

PROJECT_ROOT, OUTPUT_DIR = setup_notebook("physical_asset_building")

from physrealx import ArtifactRef, AssetSpec, PhysRealX, VersionedRef
from physrealx.integrations.physical_estimation import inspect_video
from physrealx.integrations.reconstructed_asset import reconstructed_motion

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")


Python,3.11.15
解释器,/home/yxp/miniconda3/envs/PhysReal-X/bin/python


## 1. 资产与视频

In [2]:
upstream = json.loads(
    (PROJECT_ROOT / "examples/real_assets_and_digital_twin/output/results.json").read_text()
)
reference_model = ArtifactRef.model_validate(upstream["reference-model.json"])
reconstruction_report = ArtifactRef.model_validate(upstream["reconstruction-report.json"])
registration = ArtifactRef.model_validate(upstream["reference-registration.json"])
reference = json.loads(px.artifacts.get_bytes(reference_model))
registration_document = json.loads(px.artifacts.get_bytes(registration))
reconstruction_document = json.loads(px.artifacts.get_bytes(reconstruction_report))
expected_reconstructed_motion = reconstructed_motion(
    reconstruction_document, registration_document, reference
)

saved_prior_path = OUTPUT_DIR / "vlm-physics-prior.json"
reconstructed_geometry = None
if saved_prior_path.exists():
    try:
        cached_prior = json.loads(saved_prior_path.read_text())
        candidate = ArtifactRef.model_validate(cached_prior["source_geometry"])
        cached_geometry = (
            json.loads(px.artifacts.get_bytes(candidate)) if px.artifacts.verify(candidate) else {}
        )
        if (
            cached_geometry.get("schema") == "physrealx.registered-reconstruction-geometry/4"
            and cached_geometry.get("geometry_policy") == "coordinate_transform_only"
            and cached_geometry.get("source_reconstruction")
            == reconstruction_report.model_dump(mode="json")
            and cached_geometry.get("source_registration") == registration.model_dump(mode="json")
            and cached_geometry.get("source_reference") == reference_model.model_dump(mode="json")
            and cached_geometry.get("pose_policy") == "module_one_predicted_start_pose_locked"
            and cached_geometry.get("reconstructed_motion") == expected_reconstructed_motion
            and set(cached_geometry.get("source_pose_reprojection_error_m", {}))
            == {link["name"] for link in reference["links"]}
            and max(cached_geometry["source_pose_reprojection_error_m"].values()) <= 1e-9
        ):
            reconstructed_geometry = candidate
    except (OSError, ValueError, KeyError):
        pass
if reconstructed_geometry is None:
    reconstructed_geometry = px.assets.register_reconstruction_geometry(
        reconstruction_report,
        registration,
    )
real_geometry = json.loads(px.artifacts.get_bytes(reconstructed_geometry))
capture_dir = px.assets.resolve_observation_source(
    VersionedRef.model_validate(reference["source_observations"])
)
video_dir = capture_dir / "real/video"
readiness = inspect_video(px.artifacts, video_dir)
video_status = json.loads(px.artifacts.get_bytes(readiness))
show_metrics(
    {
        **video_status["counts"],
        "几何来源": real_geometry["geometry_source"],
        "部件位姿": "锁定功能一重建 start 状态",
        "最大位姿回投误差 (nm)": 1e9
        * max(real_geometry["source_pose_reprojection_error_m"].values()),
        "视觉表面": "模块一原始重建；仅转换坐标系",
        "功能一预测 start/end (rad)": expected_reconstructed_motion["positions"],
        "碰撞验收锁定范围 (rad)": expected_reconstructed_motion["range"],
        "三角面与颜色": "原样保留",
        "部件顶点 / 三角面": ", ".join(
            f"{link}: {values['vertices']} / {values['triangles']}"
            for link, values in real_geometry["links"].items()
        ),
    }
)


depth,186
img,186
link_mask,186
mask,186
几何来源,real_artgs_reconstruction
部件位姿,锁定功能一重建 start 状态
最大位姿回投误差 (nm),0
视觉表面,模块一原始重建；仅转换坐标系
功能一预测 start/end (rad),"[0.0, 0.7912649741208384]"
碰撞验收锁定范围 (rad),"[0.0, 0.7912649741208384]"
三角面与颜色,原样保留


## 2. VLM 物性估计

方法：VLM（Qwen3-VL）。


In [3]:
images = tuple(
    px.artifacts.put_file(video_dir / "img" / f"{index}.png", media_type="image/png")
    for index in (0, 46, 93, 139, 185)
)
prior = px.assets.predict_physics(
    reference_model,
    registration,
    reconstructed_geometry,
    images,
    python="/home/yxp/miniconda3/envs/qwen/bin/python",
    model="/mnt/53dad72e-a787-4c09-961e-eff73e08888e/yxp/Desktop/ArtCot/models/Qwen3-VL-8B-Instruct",
    revision="local-content-hashed",
    timeout_seconds=1200,
    cached_prior=px.artifacts.put_file(saved_prior_path, media_type="application/json")
    if saved_prior_path.is_file()
    else None,
)
# Save the validated prior at this stage so a collision retry can reuse it.
prior_payload = px.artifacts.get_bytes(prior)
prior_temporary = saved_prior_path.with_suffix(".tmp")
prior_temporary.write_bytes(prior_payload)
prior_temporary.replace(saved_prior_path)
prediction = json.loads(prior_payload)
show_metrics(
    {
        name + " mass (kg)": (
            f"{body['mass_kg']['value']} [{body['mass_kg']['lower']}, {body['mass_kg']['upper']}]"
        )
        for name, body in prediction["prediction"]["bodies"].items()
    }
)


body mass (kg),"1.3 [1.0, 1.6]"
cover mass (kg),"0.5 [0.3, 0.7]"


## 3. 部件表面与运动联合求解

按部件执行受限 Delaunay 重网格，分别拟合两个观测状态；通过同一关节及其解析逆变换约束双向顶点、RGB、深度与轮廓。已交叉表面使用 Gaussian 排斥和三角面 Minkowski 目标联合展开，保留观测及已分离区域的约束；进入可行状态后继续拟合。原运动范围不变，连续正间隙独立验收。

视频相机经 COLMAP 对齐功能一 start/end；端点深度复用功能一训练数据。检查帧不参与求解。下方展示输入、同步前后动画和实际误差；失败候选仅作诊断。

完整场景统一处理部件遮挡；两张 GPU 分担多姿态能量和梯度，8 个常驻 CPU 进程处理姿态与自碰撞检查。方法参考 ArtMesh 与 Mesh-Untangling，属于观测约束扩展。缓存仅在输入、配置与实现一致时复用。

In [ ]:
import torch

video_preview = px.assets.preview_physics_video(video_dir)
show_animation(px.artifacts.get_bytes(video_preview))

calibration_path = OUTPUT_DIR / "video-calibration.json"
if calibration_path.is_file():
    calibration = json.loads(calibration_path.read_text())
    show_metrics(
        {
            "COLMAP 视频帧": calibration["registered_frames"],
            "检查帧重投影中位误差 (px)": calibration["check_frame_median_error_px"],
            "深度检查中位误差 (mm)": 1000 * calibration["depth_check_median_m"],
            "范围外视频帧": len(calibration["out_of_range_frames"]),
            "标定性质": "由数据估计；非独立测量真值",
        }
    )
    calibration_preview = OUTPUT_DIR / "video-colmap/alignment-preview.png"
    if calibration_preview.is_file():
        show_image(calibration_preview.read_bytes())

video_evidence_path = OUTPUT_DIR / "video-articulation-evidence.json"
video_articulation_refinement = None
video_refinement = None
effective_motion = expected_reconstructed_motion
if video_evidence_path.is_file():
    video_evidence = px.artifacts.put_file(video_evidence_path, media_type="application/json")
    saved_refinement = OUTPUT_DIR / "video-articulation-refinement.json"
    video_articulation_refinement = px.assets.refine_video_articulation(
        video_evidence,
        config={
            "hinge_band_m": 0.065,
            "surface_query_backend": "warp_cuda",
            "gpu_devices": list(range(torch.cuda.device_count())),
            "require_accelerator": True,
            "surface_query_batch_size": 65536,
            "clearance_samples": 61,
            "solver_iterations": 120,
            "active_constraints_per_pose": 400,
            "cpu_query_workers": 8,
        },
        cached_result=px.artifacts.put_file(saved_refinement, media_type="application/json")
        if saved_refinement.is_file()
        else None,
    )
    video_refinement = json.loads(px.artifacts.get_bytes(video_articulation_refinement))
    if video_refinement["status"] == "PASSED":
        effective_motion = video_refinement["optimized_motion"]

failure_reasons = []
if video_refinement:
    if not video_refinement["optimizer"]["geometry_valid"]:
        failure_reasons.append("表面有效性或观测硬约束未通过")
    if video_refinement["swept_surface_clearance"]["status"] != "PASSED":
        failure_reasons.append("完整运动区间正间隙未通过")
    if not video_refinement["heldout_images"]["passed"]:
        failure_reasons.append("独立 RGB / Mask / 深度检查未通过")
    if any(
        v["candidate"]["p90_m"]
        > v["before"]["p90_m"] + video_refinement["config"]["max_check_degradation_m"]
        for v in video_refinement["heldout"].values()
    ):
        failure_reasons.append("独立点到面拟合误差未通过")

video_metrics = {
    "视频观测": "RGB / metric depth / object mask / link mask",
    "标定证据": "已接入" if video_evidence_path.is_file() else "缺少",
    "联合优化": video_refinement["status"] if video_refinement else "BLOCKED",
    "阻断原因": "；".join(failure_reasons)
    if video_refinement
    else ", ".join(video_status["missing"]),
    "start/end 范围": effective_motion["range"],
    "碰撞代理": (
        video_refinement["collision_proxy_status"]
        if video_refinement and video_refinement["status"] == "PASSED"
        else "BLOCKED：候选验收未通过"
    ),
}
if video_refinement:
    optimizer_report = video_refinement["optimizer"]
    video_metrics["部件优化"] = "功能一原始 Mesh + 视频约束局部重建与联合分离"
    video_metrics["直接优化观测"] = "RGB / 部件 Mask / 深度 / 多帧共享运动"
    video_metrics["接触约束"] = "受观测限制的展开 + 自碰撞 CCD + 全运动区间正间隙验收"
    video_metrics["实际迭代数"] = len(optimizer_report["history"])
    video_metrics["求解结束原因"] = (
        "连续更新未被接受；以独立验收为准"
        if optimizer_report["termination"] == "no_accepted_update"
        else "达到本次迭代预算；以独立验收为准"
    )
    video_metrics["留出 RGB / Mask / 深度检查"] = video_refinement["heldout_images"]["passed"]
    video_metrics["原始资产保持不变"] = not optimizer_report["source_artifacts_mutated"]
    video_metrics["允许最大表面偏移 (mm)"] = (
        1000 * video_refinement["config"]["max_surface_displacement_m"]
    )
    video_metrics["关节轴变化 (deg)"] = video_refinement["joint"].get("axis_change_deg", 0.0)
    video_metrics["关节原点变化 (mm)"] = 1000 * video_refinement["joint"].get(
        "origin_displacement_m", 0.0
    )
    video_metrics["视觉表面扫掠"] = video_refinement["swept_surface_clearance"]["status"]
    video_metrics["完整范围间隙下界 (mm)"] = round(
        1000 * video_refinement["swept_surface_clearance"]["minimum_clearance_lower_bound_m"], 3
    )
    video_metrics["要求正间隙 (mm)"] = 1000 * video_refinement["config"]["minimum_clearance_m"]
    for name, link in video_refinement["optimizer"]["links"].items():
        observed_topology = link["remesh"].get("video_topology", {})
        video_metrics[f"{name} 视频支持的连接更新"] = observed_topology.get("status", "未触发")
        if observed_topology.get("status") == "APPLIED":
            video_metrics[f"{name} 面片替换：移除 / 新增"] = (
                observed_topology["faces_removed"], observed_topology["faces_added"]
            )
        elif observed_topology.get("status") == "REJECTED":
            video_metrics[f"{name} 连接更新拒绝原因"] = observed_topology["reason"]
        video_metrics[f"{name} 拓扑 / 自相交检查"] = (
            "通过" if link["topology"]["valid"] and not link["self_intersections"] else "未通过"
        )
        video_metrics[f"{name} 最大局部形变 (mm)"] = round(1000 * link["max_displacement_m"], 3)
    video_metrics["视频几何查询后端"] = video_refinement["surface_query"]["backend"]
    video_metrics["未达间隙要求的区间"] = video_refinement["swept_surface_clearance"][
        "violating_intervals"
    ]
    video_metrics["无法确认间隙的区间"] = video_refinement["swept_surface_clearance"][
        "unresolved_intervals"
    ]
    for name, error in video_refinement["heldout"].items():
        video_metrics[f"{name} 检查帧 P90：原始 / 候选 (mm)"] = (
            round(1000 * error["before"]["p90_m"], 3),
            round(1000 * error["candidate"]["p90_m"], 3),
        )
show_metrics(video_metrics)

if video_articulation_refinement:
    joint_previews = px.assets.preview_video_articulation(video_articulation_refinement)
    show_animation(px.artifacts.get_bytes(joint_previews["animation"]))
    show_image(px.artifacts.get_bytes(joint_previews["contact_detail"]))
    show_image(px.artifacts.get_bytes(joint_previews["metrics"]))


## 4. 原始资产与物性装配

此装配作为原始对照；下一步会单独重建联合候选的碰撞代理。失败候选只用于诊断，不替换原始资产。

In [5]:
collision_precision = {"body": 0.002, "cover": 0.0025}
source_path = OUTPUT_DIR / "initial-assembly.json"
initial_assembly = None
if source_path.is_file():
    saved = json.loads(source_path.read_text())
    source_ids = {ref["artifact_id"] for ref in saved["asset_spec"]["evidence"]}
    if (
        reconstructed_geometry.artifact_id in source_ids
        and saved["asset_spec"]["metadata"]["reconstructed_motion"] == expected_reconstructed_motion
        and saved["asset_spec"]["metadata"].get("physics_prior")
        and px.artifacts.get_bytes(
            ArtifactRef.model_validate(saved["asset_spec"]["metadata"]["physics_prior"])
        )
        == px.artifacts.get_bytes(prior)
    ):
        initial_assembly = px.artifacts.put_file(source_path, media_type="application/json")

assembly_payload = {
    "schema": "physrealx.reference-assembly-input/2",
    "resource_id": "laptop1.video-refined",
    "version": "1.0.0",
    "display_name": "Laptop reconstructed physical asset",
    "reference_model": reference_model.model_dump(mode="json"),
    "reconstructed_geometry": reconstructed_geometry.model_dump(mode="json"),
    "registration": registration.model_dump(mode="json"),
    "predicted_physics": prior.model_dump(mode="json"),
}
if initial_assembly is None:
    source_reports = []
    for name, geometry in real_geometry["geometries"].items():
        built = px.assets.build_collision(
            ArtifactRef.model_validate(geometry),
            method="coacd",
            config={
                "source_unit": "m",
                "link": name,
                "threshold_m": collision_precision[name],
                "max_parts": 256,
                "preprocessing": "coacd",
                "weld_tolerance_m": 1e-6,
            },
        )
        source_reports.append(built.output("collision_build_report"))
    initial_assembly = px.assets.assemble(
        px.artifacts.put_json(
            {
                **assembly_payload,
                "collision_reports": [r.model_dump(mode="json") for r in source_reports],
            }
        ),
        method="reference_asset_assembly",
    ).output("asset_assembly")
show_metrics(
    {
        "原始 start/end (rad)": expected_reconstructed_motion["range"],
        "物性估计方法": "VLM",
        "模块一几何": "保持原始制品",
    }
)


原始 start/end (rad),"[0.0, 0.7912649741208384]"
物性估计方法,VLM
模块一几何,保持原始制品


## 5. 候选代理重建与独立仿真检查

对通过表面分离检查的候选，按实际间隙预算自动细分并拟合碰撞代理，保留全部源三角面。MuJoCo 显式检查全部跨部件碰撞对，采用 361 个基础姿态及自适应加密，并从三个姿态进行动力学测试。要求至少 0.2 mm 正间隙；数值稳定不等于真实物性已校准。

In [6]:
from physrealx.errors import PhysRealXError

workflow_status = "FAILED"
validation_error = None
candidate_validation = None
candidate_reports = []
assembly = initial_assembly
if video_refinement and video_refinement["status"] == "PASSED":
    try:
        candidate_reports = list(px.assets.rebuild_video_collision(video_articulation_refinement))
        candidate_validation = px.assets.validate_video_articulation(
            video_articulation_refinement,
            initial_assembly,
            tuple(candidate_reports),
            config={"sweep_samples": 361, "minimum_clearance_m": 0.0002},
        )
        validation = json.loads(px.artifacts.get_bytes(candidate_validation))
        workflow_status = validation["status"]
        physics_check = validation["physics"]
        show_metrics(
            {
                "联合验收": workflow_status,
                "视觉表面验收": video_refinement["status"],
                "MuJoCo 碰撞 / 动力学": physics_check["status"],
                "最小采样间隙下界 (mm)": 1000 * physics_check["minimum_clearance_lower_bound_m"],
                "要求正间隙 (mm)": 1000 * physics_check["required_clearance_m"],
                "实际检查姿态": physics_check["sweep_strategy"]["evaluated_samples"],
                "动力学测试": physics_check["rollouts"],
            }
        )
        if workflow_status == "PASSED":
            assembly = px.assets.assemble(
                px.artifacts.put_json(
                    {
                        **assembly_payload,
                        "video_articulation_refinement": video_articulation_refinement.model_dump(
                            mode="json"
                        ),
                        "collision_reports": [r.model_dump(mode="json") for r in candidate_reports],
                    }
                ),
                method="reference_asset_assembly",
            ).output("asset_assembly")
    except PhysRealXError as error:
        validation_error = str(error)
        show_metrics({"联合验收": "FAILED", "碰撞构建或仿真错误": str(error)})
else:
    show_metrics(
        {
            "联合验收": "BLOCKED",
            "原因": "；".join(failure_reasons) if video_refinement else "缺少标定视频输入",
        }
    )


联合验收,BLOCKED
原因,表面有效性或观测硬约束未通过；完整运动区间正间隙未通过；独立 RGB / Mask / 深度检查未通过


## 6. 导出

In [7]:
asset_ref = None
exports = {
    "registered-reconstruction-geometry.json": reconstructed_geometry,
    "vlm-physics-prior.json": prior,
    "video-readiness.json": readiness,
    "video-observations.gif": video_preview,
    "initial-assembly.json": initial_assembly,
}
if video_articulation_refinement:
    exports["video-articulation-refinement.json"] = video_articulation_refinement
    exports["joint-refinement-comparison.gif"] = joint_previews["animation"]
    exports["physical-asset-preview.gif"] = joint_previews["animation"]
    exports["joint-refinement-comparison.png"] = joint_previews["comparison"]
    exports["joint-refinement-metrics.png"] = joint_previews["metrics"]
    exports["joint-refinement-contact-detail.png"] = joint_previews["contact_detail"]
if candidate_validation:
    exports["candidate-validation.json"] = candidate_validation
if workflow_status == "PASSED":
    asset = AssetSpec.model_validate(json.loads(px.artifacts.get_bytes(assembly))["asset_spec"])
    quality = px.assets.validate(asset, profile="simulation")
    assert quality.passed, quality.model_dump(mode="json")
    package = px.assets.package(assembly, method="openusd")
    exports["physical-asset.usdz"] = package.output("usd_asset_bundle")
    exports["package.json"] = package.output("asset_package_manifest")
    exports["assembly.json"] = assembly
    asset_ref = px.assets.register(asset)
results = {
    "asset": asset_ref.model_dump(mode="json") if asset_ref else None,
    "workflow_status": workflow_status,
    "preview": joint_previews["animation"].model_dump(mode="json")
    if video_articulation_refinement
    else None,
    "source_reconstruction": reconstruction_report.model_dump(mode="json"),
    "video_articulation_refinement": video_articulation_refinement.model_dump(mode="json")
    if video_articulation_refinement
    else None,
    "candidate_validation": candidate_validation.model_dump(mode="json")
    if candidate_validation
    else None,
    "independent_physical_validation": False,
}
current_validation = {
    "schema": "physrealx.physical-build-validation/1",
    "status": workflow_status,
    "video_articulation_refinement": results["video_articulation_refinement"],
    "candidate_validation": results["candidate_validation"],
    "proxy_and_dynamics": "COMPLETED" if candidate_validation else "BLOCKED",
    "reason": None
    if candidate_validation
    else validation_error or "；".join(failure_reasons) or "候选证据未齐全",
    "motion_range_preserved": video_refinement["range_preserved"] if video_refinement else True,
    "independent_physical_validation": False,
}
exports["current-validation.json"] = px.artifacts.put_json(current_validation)
if candidate_validation is None:
    exports["candidate-validation.json"] = px.artifacts.put_json(
        {
            "schema": "physrealx.articulation-candidate-validation/1",
            "status": "BLOCKED",
            "reason": current_validation["reason"],
            "source_refinement": results["video_articulation_refinement"],
            "physics": None,
            "published_asset": False,
        }
    )
if workflow_status != "PASSED":
    for stale in ("physical-asset.usdz", "assembly.json", "package.json"):
        (OUTPUT_DIR / stale).unlink(missing_ok=True)
px.artifacts.export(OUTPUT_DIR, results, files=exports)
print(OUTPUT_DIR / "results.json")


/mnt/53dad72e-a787-4c09-961e-eff73e08888e/yxp/Desktop/pingtaidajian/PhysReal-X/examples/physical_asset_building/output/results.json
